In [1]:
%pip install datasets

   ---------------------------------------- 0.0/27.6 MB ? eta -:--:--
   ------------ --------------------------- 8.4/27.6 MB 43.2 MB/s eta 0:00:01
   ------------------------------- -------- 21.8/27.6 MB 52.9 MB/s eta 0:00:01
   ---------------------------------------- 27.6/27.6 MB 48.6 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
import urllib.request
import tarfile
import os


url = "https://raw.githubusercontent.com/bepnye/EBM-NLP/master/ebm_nlp_2_00.tar.gz"
urllib.request.urlretrieve(url, "ebm_nlp_2_00.tar.gz")

with tarfile.open("ebm_nlp_2_00.tar.gz", "r:gz") as tar:
    tar.extractall("ebm_nlp_data")

os.remove("ebm_nlp_2_00.tar.gz")
print("Done! The raw data is ready in the 'ebm_nlp_data/ebm_nlp_2_00' folder.")

C:\Users\erank\AppData\Local\Temp\ipykernel_21696\2489365289.py:10: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall("ebm_nlp_data")


Done! The raw data is ready in the 'ebm_nlp_data/ebm_nlp_2_00' folder.


In [6]:
import os

def run_diagnostics(base_dir="ebm_nlp_data/ebm_nlp_2_00"):
    print("--- PATH DIAGNOSTICS ---")
    print(f"Current Working Directory: {os.getcwd()}")
    
    
    documents_dir = os.path.join(base_dir, "documents")
    print(f"Checking Documents: {documents_dir}")
    print(f"Exists: {os.path.exists(documents_dir)}\n")
    
    
    base_ann_dir = os.path.join(base_dir, "annotations", "aggregated", "starting_spans")
    print(f"Checking Base Annotations: {base_ann_dir}")
    print(f"Exists: {os.path.exists(base_ann_dir)}\n")
    
    
    sub_paths = {"train": "train", "test": os.path.join("test", "gold")}
    
    for split_name, sub_path in sub_paths.items():
        p_dir = os.path.join(base_ann_dir, "participants", sub_path)
        print(f"Checking {split_name} split at: {p_dir}")
        
        if not os.path.exists(p_dir):
            print("--> FAILED: Directory not found.\n")
            continue
            
        files = os.listdir(p_dir)
        print(f"--> SUCCESS: Directory found with {len(files)} files.")
        
        
        for ann_file in files:
            if ann_file.endswith(".ann"):
                pmid = ann_file.replace(".ann", "")
                token_file = os.path.join(documents_dir, f"{pmid}.tokens")
                print(f"--> Sample check for token file: {token_file}")
                print(f"--> Token file exists: {os.path.exists(token_file)}\n")
                break

run_diagnostics()

--- PATH DIAGNOSTICS ---
Current Working Directory: c:\Users\erank\Desktop\Biomedical LLM Clarification Seeking\Biomedical-LLM-Clarification-Seeking
Checking Documents: ebm_nlp_data/ebm_nlp_2_00\documents
Exists: True

Checking Base Annotations: ebm_nlp_data/ebm_nlp_2_00\annotations\aggregated\starting_spans
Exists: True

Checking train split at: ebm_nlp_data/ebm_nlp_2_00\annotations\aggregated\starting_spans\participants\train
--> SUCCESS: Directory found with 4792 files.
--> Sample check for token file: ebm_nlp_data/ebm_nlp_2_00\documents\10036953.AGGREGATED.tokens
--> Token file exists: False

Checking test split at: ebm_nlp_data/ebm_nlp_2_00\annotations\aggregated\starting_spans\participants\test\gold
--> SUCCESS: Directory found with 189 files.
--> Sample check for token file: ebm_nlp_data/ebm_nlp_2_00\documents\10070173.AGGREGATED.tokens
--> Token file exists: False



In [ ]:
import os
import json

def load_ebm_nlp_data(base_dir="ebm_nlp_data/ebm_nlp_2_00"):
    documents_dir = os.path.join(base_dir, "documents")
    base_ann_dir = os.path.join(base_dir, "annotations", "aggregated", "starting_spans")
    
    parsed_data = []
    
    sub_paths = {
        "train": "train",
        "test": os.path.join("test", "gold")
    }
    
    elements = {"P": "participants", "I": "interventions", "O": "outcomes"}
    
    for split_name, sub_path in sub_paths.items():
        # Find all PMIDs for this split by looking inside the participants folder
        p_dir = os.path.join(base_ann_dir, "participants", sub_path)
        
        if not os.path.exists(p_dir):
            continue
            
        for ann_file in os.listdir(p_dir):
            if not ann_file.endswith(".ann"): 
                continue
            
            
            pmid = ann_file.split(".")[0] 
            token_file = os.path.join(documents_dir, f"{pmid}.tokens")
            
            if not os.path.exists(token_file):
                continue
                
            
            with open(token_file, 'r', encoding='utf-8') as f:
                tokens = f.read().split()
            
            abstract_data = {
                "pmid": pmid,
                "split": split_name,  
                "text": " ".join(tokens),
                "spans": {"P": [], "I": [], "O": []}
            }
            
            
            for slot, folder_name in elements.items():
                
                slot_ann_file = os.path.join(base_ann_dir, folder_name, sub_path, ann_file) 
                
                if os.path.exists(slot_ann_file):
                    with open(slot_ann_file, 'r', encoding='utf-8') as f:
                        labels = f.read().split()
                    
                    current_span = []
                    for token, label in zip(tokens, labels):
                        if label.strip() == '1':
                            current_span.append(token)
                        elif label.strip() == '0' and current_span:
                            abstract_data["spans"][slot].append(" ".join(current_span))
                            current_span = []
                    
                    if current_span:
                        abstract_data["spans"][slot].append(" ".join(current_span))
            
            parsed_data.append(abstract_data)
            
    return parsed_data

dataset = load_ebm_nlp_data()


{
  "pmid": "10036953",
  "split": "train",
  "text": "[ Triple therapy regimens involving H2 blockaders for therapy of Helicobacter pylori infections ] . Comparison of ranitidine and lansoprazole in short-term low-dose triple therapy for Helicobacter pylori infection . To evaluate the efficacy and safety of two 1-week low-dose triple-therapy drug regimens involving antisecretory drugs for Helicobacter pylori infection , 99 patients with H. pylori infection were treated with either lansoprazole ( LPZ ) or ranitidine ( RNT ) used together with clarithromycin ( CAM ) and metrinidazole ( MTZ ) . The drug combination and administration periods in the PPI group were LPZ 30 mg , CAM 400 mg , MTZ 500 mg ( LCM group ) . The ranitidine group received RNT 300 mg , CAM 400 mg , MTZ 500 mg ( RCM group ) . The cure rate of H. pylori infection was 88 % in the LCM group ; 95 % CI 79-97 and 92 % in the RCM group ; 95 % CI 84-99 .",
  "spans": {
    "P": [
      "99 patients with H. pylori infection we

In [11]:

output_filename = "ebm_nlp_ground_truth.json"

with open(output_filename, "w", encoding="utf-8") as f:
    json.dump(dataset, f, indent=2)

print(f"\nSuccessfully saved all {len(dataset)} parsed abstracts to {output_filename}")



Successfully saved all 4981 parsed abstracts to ebm_nlp_ground_truth.json


In [12]:
import json

with open("ebm_nlp_ground_truth.json", "r", encoding="utf-8") as f:
    ground_truth_data = json.load(f)

print(f"Loaded {len(ground_truth_data)} abstracts. Ready to apply masking!")

Loaded 4981 abstracts. Ready to apply masking!


In [1]:
import json
import spacy
import random


random.seed(42)


print("Loading spaCy model...")
nlp = spacy.load("en_core_web_sm")

def create_and_save_masked_data(input_file="ebm_nlp_ground_truth.json", output_file="ebm_nlp_masked_test.json"):
    
    print(f"Loading ground truth data from {input_file}...")
    with open(input_file, "r", encoding="utf-8") as f:
        dataset = json.load(f)
        
    
    test_data = [d for d in dataset if d.get('split') == 'test']
    print(f"Found {len(test_data)} gold-standard test abstracts.")
    
    ambiguous_examples = []
    
    for abstract_data in test_data:
        
        available_slots = [slot for slot, spans in abstract_data['spans'].items() if spans]
        if not available_slots:
            continue
            
        
        target_slot = random.choice(available_slots)
        spans_to_remove = abstract_data['spans'][target_slot]
        
        
        doc = nlp(abstract_data['text'])
        sentences = [sent.text.strip() for sent in doc.sents]
        
        masked_abstract = []
        withheld_sentences = []
        
        for sentence in sentences:
            
            contains_span = any(span.lower() in sentence.lower() for span in spans_to_remove)
            
            if contains_span:
                withheld_sentences.append(sentence)
            else:
                masked_abstract.append(sentence)
        
       
        if not withheld_sentences:
            continue
            
        ambiguous_examples.append({
            "pmid": abstract_data['pmid'],
            "missing_slot": target_slot,
            "original_text": abstract_data['text'],
            "masked_text": " ".join(masked_abstract),
            "withheld_text": " ".join(withheld_sentences),
            "target_spans": spans_to_remove
        })
        
    
    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(ambiguous_examples, f, indent=2)
        
    print(f"Successfully generated and saved {len(ambiguous_examples)} ambiguous examples to {output_file}")
    
    
    if ambiguous_examples:
        print("\n--- Sample Output ---")
        print(f"PMID: {ambiguous_examples[0]['pmid']}")
        print(f"Missing Slot: {ambiguous_examples[0]['missing_slot']}")
        print(f"Withheld Text: {ambiguous_examples[0]['withheld_text']}")


create_and_save_masked_data()

Loading spaCy model...
Loading ground truth data from ebm_nlp_ground_truth.json...
Found 189 gold-standard test abstracts.
Successfully generated and saved 189 ambiguous examples to ebm_nlp_masked_test.json

--- Sample Output ---
PMID: 10070173
Missing Slot: O
Withheld Text: Secondarily to ascertain patients ' preferences for the two nasal devices and to assess quality of life . RESULTS Mean daily nasal symptom scores were significantly reduced with treatment . There were no statistically significant changes from baseline for eye symptoms . The most common nasal and non-nasal adverse events for both groups were epistaxis and headache . Turbuhaler was easier to use and more convenient to carry , had less of an unpleasant taste , and caused less nasal irritation than the aqua spray . Improvement in quality of life from baseline to clinic visits was statistically significant in both groups . Once daily use of 256 mg of budesonide aqua and 400 mg of budesonide Turbuhaler are equally safe a